# Week 73, Day 4: Versioning and changelogs

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w073d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
versions = ["1.9.0", "1.10.0", "1.2.0", "2.0.0rc1", "2.0.0"]
print(sorted(versions))
print(max(versions))

def key(v):
    return tuple(int(p) for p in v.split(".") if p.isdigit())

print(sorted(versions[:3], key=key))

In [ ]:
import functools, warnings

def deprecated(instead, removal):
    def decorate(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            warnings.warn(f"{func.__name__}() is deprecated and will be removed in {removal}; use {instead}() instead",
                          DeprecationWarning, stacklevel=2)
            return func(*args, **kwargs)
        return wrapper
    return decorate

def word_count(text):
    return len(text.split())

@deprecated("word_count", "3.0")
def count_words(text):
    return word_count(text)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print(count_words("the old way still works"))
print(caught[0].category.__name__, "-", caught[0].message)

> **Key idea.** SemVer's MAJOR.MINOR.PATCH promise: patch for fixes, minor for compatible features (and deprecations), major for breaking changes; 0.x may change anything; pre-releases come before their release; build metadata is ignored. PEP 440 spells versions as release numbers with optional aN, bN, rcN, .postN and .devN (dev < a < b < rc < release < post); compare them as parsed values, never as strings. Keep one source of the version; read it with importlib.metadata.version. Deprecate with DeprecationWarning (stacklevel=2) in a minor release, remove in a major one. Changelogs list Added, Changed, Deprecated, Removed, Fixed and Security per release; conventional commits (feat, fix, ! or BREAKING CHANGE) drive the next version.

### Exercise: Comparing semantic versions

Write `parse(version)`: split a SemVer string into `(major, minor, patch, pre)`, where `pre` is a tuple of pre-release identifiers (ints where they're all digits, otherwise strings), or `()`; drop any build metadata after `+`. Raise `ValueError` if the core isn't three non-negative whole numbers. Then `key(version)`: a sort key that orders versions by the SemVer rules (a release after its pre-releases; identifiers compared numbers before words, numbers numerically, words alphabetically; fewer identifiers first). Finally `latest(versions, allow_pre=False)`: the highest version, ignoring pre-releases unless allowed (or `None`).

In [ ]:
def parse(version):
    ___

def key(version):
    ___

def latest(versions, allow_pre=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_semver():\n    \"SemVer precedence\"\n    assert parse(\"1.10.2\") == (1, 10, 2, ()) and parse(\"3.0.0-rc.2+build.7\") == (3, 0, 0, (\"rc\", 2))\n    order = [\"1.0.0-alpha\", \"1.0.0-alpha.1\", \"1.0.0-alpha.beta\", \"1.0.0-beta\", \"1.0.0-beta.2\", \"1.0.0-beta.11\", \"1.0.0-rc.1\", \"1.0.0\", \"1.2.0\", \"1.10.0\", \"2.0.0\"]\n    shuffled = order[5:] + order[:5][::-1]\n    assert sorted(shuffled, key=key) == order\n    assert key(\"1.0.0+abc\") == key(\"1.0.0\")\n    assert latest([\"1.9.0\", \"1.10.0\", \"2.0.0-rc.1\"]) == \"1.10.0\" and latest([\"1.9.0\", \"2.0.0-rc.1\"], allow_pre=True) == \"2.0.0-rc.1\"\n    assert latest([\"1.0.0-rc.1\"]) is None\n    for bad in [\"1.0\", \"1.0.x\", \"v1.0.0\", \"1.-1.0\"]:\n        try:\n            parse(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: The next number

Write `bump(version, part)` for SemVer versions without build metadata: `part` is `"major"`, `"minor"` or `"patch"`, resetting the smaller numbers. A pre-release is **finished** by the bump that reaches it: `bump("2.0.0-rc.1", "major")` is `"2.0.0"` (because 2.0.0 is the next major after the 1.x it follows), as is `"minor"` or `"patch"` on it. Also `pre(version, label)`: start or continue a pre-release: `pre("1.4.2", "rc")` is `"1.4.3-rc.1"`, `pre("1.4.3-rc.1", "rc")` is `"1.4.3-rc.2"`, and `pre("1.4.3-beta.2", "rc")` is `"1.4.3-rc.1"`.

In [ ]:
def bump(version, part):
    ___

def pre(version, label):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bump():\n    \"bumping by the rules\"\n    assert bump(\"1.4.2\", \"patch\") == \"1.4.3\" and bump(\"1.4.2\", \"minor\") == \"1.5.0\" and bump(\"1.4.2\", \"major\") == \"2.0.0\"\n    assert bump(\"0.9.9\", \"minor\") == \"0.10.0\"\n    assert bump(\"2.0.0-rc.1\", \"major\") == \"2.0.0\" and bump(\"2.0.0-rc.1\", \"patch\") == \"2.0.0\"\n    assert bump(\"2.1.0-rc.1\", \"major\") == \"3.0.0\" and bump(\"2.1.0-rc.1\", \"minor\") == \"2.1.0\"\n    assert pre(\"1.4.2\", \"rc\") == \"1.4.3-rc.1\" and pre(\"1.4.3-rc.1\", \"rc\") == \"1.4.3-rc.2\" and pre(\"1.4.3-beta.2\", \"rc\") == \"1.4.3-rc.1\"\n    try:\n        bump(\"1.0.0\", \"huge\")\n        assert False\n    except ValueError:\n        pass")

### Exercise: Sorting Python versions

Write `pep440_key(version)` for versions of the form `N(.N)*[{a|b|rc}N][.postN][.devN]` (no epochs or local labels; parse with a regular expression, and raise `ValueError` otherwise). Trailing zeros in the release don't matter (`1.4 == 1.4.0`). The order for one release is `.devN` of the final release, then `aN`, `bN`, `rcN` (each possibly with its own `.devN` before it), then the release, then `.postN`. Then `newest(versions, pre=False)`: the highest, skipping pre-releases and dev releases unless `pre` is true.

In [ ]:
import re

PATTERN = re.compile(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?(?:\.post(\d+))?(?:\.dev(\d+))?")

def pep440_key(version):
    ___

def newest(versions, pre=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pep440():\n    \"PEP 440 ordering\"\n    order = [\"1.4.dev1\", \"1.4a1.dev1\", \"1.4a1\", \"1.4b2\", \"1.4rc1\", \"1.4\", \"1.4.post1.dev1\", \"1.4.post1\", \"1.4.1\", \"1.10\"]\n    assert sorted(reversed(order), key=pep440_key) == order\n    assert pep440_key(\"1.4\") == pep440_key(\"1.4.0\") == pep440_key(\"1.4.0.0\")\n    assert newest([\"1.9\", \"1.10rc1\", \"1.10.dev3\"]) == \"1.9\" and newest([\"1.9\", \"1.10rc1\"], pre=True) == \"1.10rc1\"\n    assert newest([\"2.0.post1\", \"2.0\"]) == \"2.0.post1\"\n    for bad in [\"1.4-rc1\", \"v1.4\", \"1.4c1\", \"\"]:\n        try:\n            pep440_key(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: Which bump?

Write `release_type(message)` for a conventional commit message: `"major"` if the type is followed by `!` (as in `feat!:` or `fix(api)!:`) or any line of the message starts with `BREAKING CHANGE:`; `"minor"` for `feat`; `"patch"` for `fix` and `perf`; otherwise `None`. Then `next_version(current, messages)`: apply the **largest** release type among the messages once, to a SemVer `current`, except that while the major number is `0`, a breaking change bumps only the minor number. Return `current` unchanged if no message calls for a release.

In [ ]:
import re

def release_type(message):
    ___

def next_version(current, messages):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_commits():\n    \"conventional commits\"\n    assert release_type(\"feat: add word_count\") == \"minor\" and release_type(\"fix(parser): empty input\") == \"patch\"\n    assert release_type(\"feat(api)!: rename load\") == \"major\" and release_type(\"refactor: tidy\\n\\nBREAKING CHANGE: load() renamed\") == \"major\"\n    assert release_type(\"docs: typo\") is None and release_type(\"perf: faster counting\") == \"patch\" and release_type(\"not conventional\") is None\n    assert next_version(\"1.4.2\", [\"fix: a\", \"docs: b\", \"feat: c\"]) == \"1.5.0\"\n    assert next_version(\"1.4.2\", [\"fix: a\", \"feat!: c\"]) == \"2.0.0\"\n    assert next_version(\"0.4.2\", [\"feat!: c\"]) == \"0.5.0\" and next_version(\"0.4.2\", [\"fix: x\"]) == \"0.4.3\"\n    assert next_version(\"1.4.2\", [\"chore: deps\", \"test: more\"]) == \"1.4.2\"")

### Exercise: Drafting a changelog

Write `changelog_section(version, day, messages)`: the Keep a Changelog section for one release, from conventional commit messages. The heading is `f"## [{version}] - {day}"`. Group the first lines' descriptions (the text after `: `) into `### Added` (feat), `### Changed` (perf and refactor), `### Fixed` (fix) and `### Removed` (a description starting with `remove`, of any type, goes here instead), in that order, skipping empty groups, with each entry as `- Description` (first letter capitalised) in the order given; and add `**Breaking:** ` before the description of any breaking change. Other types are left out. Lines are joined with newlines, with a blank line after the heading, and nothing at the end.

In [ ]:
import re

def changelog_section(version, day, messages):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_section():\n    \"a changelog from commits\"\n    msgs = [\"feat: word_count accepts a minimum\", \"fix(parser): crash on empty input\", \"docs: fix typo\",\n            \"perf: faster counting\", \"feat!: remove count_words\", \"refactor: tidy the tokeniser\"]\n    assert changelog_section(\"2.0.0\", \"2026-11-10\", msgs) == (\n        \"## [2.0.0] - 2026-11-10\\n\\n\"\n        \"### Added\\n- Word_count accepts a minimum\\n\"\n        \"### Changed\\n- Faster counting\\n- Tidy the tokeniser\\n\"\n        \"### Fixed\\n- Crash on empty input\\n\"\n        \"### Removed\\n- **Breaking:** Remove count_words\"\n    )\n    assert changelog_section(\"1.0.1\", \"2026-11-11\", [\"fix: one thing\"]) == \"## [1.0.1] - 2026-11-11\\n\\n### Fixed\\n- One thing\"")

### Exercise: Deprecating gently

Write a decorator factory `deprecated(instead, removal)` that makes a function emit a `DeprecationWarning` with the message `"NAME() is deprecated and will be removed in REMOVAL; use INSTEAD() instead"` each time it's called, pointing at the **caller** (`stacklevel=2`), and then behave exactly as before (keep its name). Then `renamed(old_name, new_function, removal)`: return a deprecated alias for `new_function` whose name is `old_name`.

In [ ]:
import functools, warnings

def deprecated(instead, removal):
    ___

def renamed(old_name, new_function, removal):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_deprecation():\n    \"warnings that point at the caller\"\n    import warnings\n    @deprecated(\"word_count\", \"3.0\")\n    def count_words(text):\n        return len(text.split())\n    with warnings.catch_warnings(record=True) as caught:\n        warnings.simplefilter(\"always\")\n        assert count_words(\"a b c\") == 3\n        line = __import__(\"sys\")._getframe().f_lineno - 1\n    assert len(caught) == 1 and caught[0].category is DeprecationWarning\n    assert str(caught[0].message) == \"count_words() is deprecated and will be removed in 3.0; use word_count() instead\"\n    assert caught[0].lineno == line, \"the warning should point at the caller's line (stacklevel=2)\"\n    assert count_words.__name__ == \"count_words\"\n    def word_count(text):\n        return len(text.split())\n    old = renamed(\"countWords\", word_count, \"3.0\")\n    with warnings.catch_warnings(record=True) as caught:\n        warnings.simplefilter(\"always\")\n        assert old(\"x y\") == 2\n    assert old.__name__ == \"countWords\" and \"countWords() is deprecated\" in str(caught[0].message) and \"use word_count()\" in str(caught[0].message)")

### Bug hunt: Versions compared as text

The versions are compared as strings, so `"1.10.0"` looks older than `"1.9.0"` and the program says no upgrade is needed. Compare them as tuples of numbers so this prints `upgrade to 1.10.0`.

In [ ]:
installed, available = "1.9.0", "1.10.0"
if available > installed:
    print(f"upgrade to {available}")
else:
    print("up to date")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"numeric comparison\"\n    expect_output(\"upgrade to 1.10.0\")")

### Bug hunt: A minor bump that kept the patch

A minor release should reset the patch number to zero, but `bump_minor` keeps it, so this prints `1.5.2`. Fix it so it prints `1.5.0`.

In [ ]:
def bump_minor(version):
    major, minor, patch = (int(x) for x in version.split("."))
    return f"{major}.{minor + 1}.{patch}"

print(bump_minor("1.4.2"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"reset the smaller number\"\n    expect_output(\"1.5.0\")")

### Bug hunt: A release candidate after its release

The key treats a pre-release as an **extra** part after the numbers, so `2.0.0-rc.1` sorts after `2.0.0` and is reported as the newest. Make a release sort after its own pre-releases, so this prints `['1.9.0', '2.0.0-rc.1', '2.0.0'] 2.0.0`.

In [ ]:
def key(v):
    core, _, pre = v.partition("-")
    numbers = tuple(int(x) for x in core.split("."))
    return numbers + ((pre,) if pre else ())

versions = ["2.0.0", "1.9.0", "2.0.0-rc.1"]
ordered = sorted(versions, key=key)
print(ordered, ordered[-1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"pre-releases first\"\n    expect_output(\"['1.9.0', '2.0.0-rc.1', '2.0.0'] 2.0.0\")")

### Puzzle: A changelog entry

Arrange the lines of a Keep a Changelog entry for version 2.4.0, under the Unreleased section. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the next version after 2.3.1 for a backwards-compatible new feature
q = ___   # True or False: in SemVer, 3.0.0-rc.2 is greater than 3.0.0
r = ___   # sorted(["1.4", "1.4rc1", "1.4.post1", "1.4.dev1"], key=pep440 order), as a list
s = ___   # the warning category for something that will be removed in a future version
t = ___   # the release type for the commit "fix(io)!: open files in binary mode"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"2.4.0\" and q is False and r == [\"1.4.dev1\", \"1.4rc1\", \"1.4\", \"1.4.post1\"], \"p, q and r\"\n    assert s in (\"DeprecationWarning\", DeprecationWarning) and t == \"major\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: cutting a release

Write `cut_release(pyproject, changelog, messages, day)`. Read the current version from `pyproject` with `tomllib` (`project.version`), work out the next version from the conventional commit `messages` (as in the practice: the largest type wins, breaking bumps minor while the major number is 0; `next_version` and `changelog_section` are given in the starter), and return `(version, new_pyproject, new_changelog)`. If no release is needed, return `(None, pyproject, changelog)` unchanged. In `new_pyproject`, replace only the `version = "..."` line inside `[project]` (other tables may have a `version` key too). In `new_changelog`, insert the new section straight after the `## [Unreleased]` line, with a blank line before and after it (so Unreleased becomes empty and the old releases follow); if there's no Unreleased line, raise `ValueError`.

In [ ]:
import re, tomllib

def next_version(current, messages):
    rank = {"patch": 1, "minor": 2, "major": 3}
    kinds = []
    for message in messages:
        lines = message.splitlines() or [""]
        m = re.match(r"(\w+)(\([^)]*\))?(!)?:", lines[0])
        if not m:
            continue
        if m.group(3) or any(l.startswith("BREAKING CHANGE:") for l in lines):
            kinds.append("major")
        elif m.group(1) == "feat":
            kinds.append("minor")
        elif m.group(1) in ("fix", "perf"):
            kinds.append("patch")
    if not kinds:
        return current
    kind = max(kinds, key=rank.get)
    major, minor, patch = (int(x) for x in current.split("."))
    if kind == "major" and major == 0:
        kind = "minor"
    return {"major": f"{major + 1}.0.0", "minor": f"{major}.{minor + 1}.0", "patch": f"{major}.{minor}.{patch + 1}"}[kind]

def changelog_section(version, day, messages):
    groups = {"Added": [], "Changed": [], "Fixed": []}
    for message in messages:
        m = re.match(r"(\w+)(\([^)]*\))?(!)?: (.*)", message.splitlines()[0])
        if m:
            group = {"feat": "Added", "perf": "Changed", "refactor": "Changed", "fix": "Fixed"}.get(m.group(1))
            if group:
                text = m.group(4)
                groups[group].append(("**Breaking:** " if m.group(3) else "") + text[:1].upper() + text[1:])
    out = [f"## [{version}] - {day}", ""]
    for name, entries in groups.items():
        if entries:
            out.append(f"### {name}")
            out.extend(f"- {e}" for e in entries)
    return "\n".join(out)

def cut_release(pyproject, changelog, messages, day):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_release():\n    \"a version bump and a changelog entry\"\n    pyproject = '[project]\\nname = \"textkit\"\\nversion = \"1.4.2\"\\n\\n[tool.other]\\nversion = \"9.9\"\\n'\n    changelog = \"# Changelog\\n\\n## [Unreleased]\\n\\n## [1.4.2] - 2026-10-01\\n### Fixed\\n- Old fix\\n\"\n    version, new_py, new_cl = cut_release(pyproject, changelog, [\"fix: crash on empty\", \"feat: minimum length\"], \"2026-11-10\")\n    assert version == \"1.5.0\"\n    assert new_py == '[project]\\nname = \"textkit\"\\nversion = \"1.5.0\"\\n\\n[tool.other]\\nversion = \"9.9\"\\n'\n    assert new_cl == (\"# Changelog\\n\\n## [Unreleased]\\n\\n## [1.5.0] - 2026-11-10\\n\\n### Added\\n- Minimum length\\n### Fixed\\n- Crash on empty\\n\\n\"\n                      \"## [1.4.2] - 2026-10-01\\n### Fixed\\n- Old fix\\n\")\n    assert cut_release(pyproject, changelog, [\"docs: x\"], \"2026-11-10\") == (None, pyproject, changelog)\n    try:\n        cut_release(pyproject, \"# Changelog\\n\", [\"fix: x\"], \"2026-11-10\")\n        assert False\n    except ValueError:\n        pass")

## Warm-up

As strings, `"1.10.0"` sorts **before** `"1.9.0"` (the character `1` is less than `9`), and `max` thinks `2.0.0rc1` is newer than `2.0.0`. Versions are sequences of numbers with special parts, and comparing them needs real rules.

## Semantic versioning

**Semantic versioning** (SemVer) gives the three numbers of `MAJOR.MINOR.PATCH` meanings tied to the **public interface** (the functions, classes and behaviour that users rely on):

- **PATCH** goes up for backwards-compatible **bug fixes**: `2.3.1` to `2.3.2`.
- **MINOR** goes up for backwards-compatible **new features** (and deprecations), resetting PATCH: `2.3.2` to `2.4.0`.
- **MAJOR** goes up for **breaking changes** (anything that could make working user code fail), resetting the others: `2.4.0` to `3.0.0`.
- **`0.y.z`** means initial development: anything may change at any time. Many projects treat a minor bump in `0.x` as breaking.
- **Pre-releases** add a hyphen and dot-separated identifiers: `3.0.0-alpha.1`, `3.0.0-rc.2`. They come **before** the release: `3.0.0-rc.2 < 3.0.0`. Identifiers are compared one by one: numbers numerically, words alphabetically, a number before a word, and fewer identifiers first when the rest are equal.
- **Build metadata** after a `+` (`3.0.0+build.17`) is ignored when comparing.

What counts as breaking? Removing or renaming anything public, adding a required argument, changing a return type or a default, raising different exceptions, and dropping support for a Python version. Deciding what's public is half the job: `__all__`, the documentation, and the `_` prefix (Day 1) draw the line.

**Calendar versioning** (CalVer), such as `2026.11.0` (used by pip itself), dates releases instead. It suits tools and applications more than libraries, because a date says nothing about compatibility.

## Python's version scheme

Python packages use **PEP 440**, which is close to SemVer but spelt differently, and has a few more parts. In order of appearance:

```text
[N!]N(.N)*[{a|b|rc}N][.postN][.devN][+local]
```

- the **release** numbers, any number of them: `1.4`, `1.4.2`, `2026.11`;
- an optional **pre-release**: `1.4.0a1` (alpha), `1.4.0b2` (beta), `1.4.0rc1` (release candidate);
- an optional **post-release**: `1.4.0.post1`, for fixes that don't change the code (a packaging mistake, a documentation fix);
- an optional **development release**: `1.4.0.dev3`, a snapshot before the version is finished;
- rarely, an **epoch** (`1!2.0`, for a project that changed its numbering scheme) and a **local version label** (`+ubuntu1`), which can't be uploaded to PyPI.

The ordering for one release: `1.4.0.dev1 < 1.4.0a1 < 1.4.0b1 < 1.4.0rc1 < 1.4.0 < 1.4.0.post1`. Trailing zeros don't matter: `1.4` equals `1.4.0`. pip **ignores pre-releases** when choosing what to install unless you ask (`pip install --pre`) or a specifier names one. The `packaging` library (which pip uses) implements all this as `packaging.version.Version`; today you'll build the core of it.

**Where the version lives.** Keep **one** source of truth: either `version` in `pyproject.toml`, or a dynamic version taken from the code or a Git tag. At run time, read the installed version with `importlib.metadata.version("textkit")` rather than maintaining a second copy by hand.

## Deprecation

You can't remove something users rely on without warning them. The standard path:

1. In a **minor** release, keep the old name working but make it emit a **`DeprecationWarning`** (with `warnings.warn(message, DeprecationWarning, stacklevel=2)`, where `stacklevel=2` points the warning at the **caller's** line, not yours), saying what to use instead and when it will go. Document it in the changelog under "Deprecated".
2. Keep it for at least one minor release (often longer: say how long in your documentation).
3. Remove it in the next **major** release, under "Removed".

`DeprecationWarning` is hidden by default, except for code run directly as `__main__` and in test runs: pytest shows them in its summary, and `python -W error::DeprecationWarning` turns them into errors, which is a good way to find out what an upgrade will break.

## Changelogs and commit messages

A **changelog** (`CHANGELOG.md`) tells users what changed in each release, in **their** terms: not "refactored the parser" but "parsing is twice as fast" or "`load()` now raises `FileNotFoundError` for missing files". The **Keep a Changelog** convention, newest release first:

```markdown
# Changelog

## [Unreleased]

## [2.4.0] - 2026-11-10
### Added
- `word_count()` accepts a `minimum` length.
### Deprecated
- `count_words()`: use `word_count()`. It will be removed in 3.0.
### Fixed
- Counting no longer crashes on an empty string.
```

with sections **Added**, **Changed**, **Deprecated**, **Removed**, **Fixed** and **Security**, and an **Unreleased** section collecting changes as they're merged.

**Conventional Commits** make commit messages machine-readable, so tools can choose the next version and draft the changelog: `type(scope)!: description`, where `type` is `feat` (a feature: minor), `fix` (a bug fix: patch), or `docs`, `refactor`, `test`, `chore` and so on (no release on their own); a `!` after the type, or a `BREAKING CHANGE:` line in the message body, marks a breaking change (major). Tag each release in Git (`git tag v2.4.0`), so the commits since the last release are easy to find.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the changelog of a library you use (most have a `CHANGELOG.md` or a "Release notes" page) and find a deprecation, from its warning to its removal. Tomorrow: **dependency management**: version specifiers, pinning, lock files, and how a resolver finds versions of everything that fit together.